In [0]:
%sql
-- ============================================================
-- PASO 2: Crear el esquema Landing
-- ============================================================
-- Crear catalogo si no existe
Create catalog bootcamp;
-- Usar tu catálogo
USE CATALOG bootcamp;
-- Crear esquema Landing
CREATE SCHEMA IF NOT EXISTS bootcamp.landing
COMMENT 'Esquema para archivos';
-- Crear esquema Raw
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze
COMMENT 'Esquema para datos crudos sin procesar';
-- Verificar
SHOW SCHEMAS;

In [0]:
%sql
-- ============================================================
-- PASO 3: Crear the Volume para archivos
-- ============================================================
-- Crear volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';
-- Verificar que se creó
SHOW VOLUMES IN bootcamp.landing;

In [0]:
%sql
-- ============================================================
-- PASO 4: Verificar que el archivo se subió correctamente
-- ============================================================
-- Listar archivos en el volume
LIST '/Volumes/bootcamp/landing/archivos/';


In [0]:
%sql
-- Leer directo del archivo es posible, y es útil también para analizarlo
SELECT * FROM read_files(
 '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
 format => 'csv',
 header => true
 )

In [0]:
%sql
-- ============================================================
-- PASO 5: Crear la tabla properties_bronze
-- ============================================================
-- Primero, eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;
-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
CREATE TABLE bootcamp.bronze.properties_bronze
 SELECT * FROM read_files(
 '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
 format => 'csv',
 header => true
 )
 where url like 'https%' --Filtramos por aquellos links que sean consistentes
 ;

In [0]:
%sql
-- Verificar que la tabla se creó correctamente
SHOW TABLES IN bootcamp.bronze;

PARTE 1: Exploración Inicial

Ejercicio 1.1: Verificar datos disponibles - ¿Cuántos registros hay en total en la tabla
bootcamp.bronze.properties_bronze? Pista: Usa COUNT(*)

Ejercicio 1.2: Explorar estructura - ¿Qué columnas tiene la tabla? ¿Qué tipos de datos tienen? Pista: Usa
DESCRIBE TABLE o SELECT * LIMIT 0

Ejercicio 1.3: Ver muestra de datos - Muestra las primeras 10 filas del dataset, pero solo las columnas más
importantes: id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales,
antiguedad, estado, zona. Pista: Usa SELECT con columnas específicas y LIMIT 10

In [0]:
%sql
Select count(*) from bootcamp.bronze.properties_bronze

In [0]:
%sql
describe table bootcamp.bronze.properties_bronze

In [0]:
%sql
Select id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales, antiguedad, estado, zona
from bootcamp.bronze.properties_bronze
limit 10

PARTE 2: Análisis de Valores Nulos

Ejercicio 2.1: Contar nulos por columna - ¿Cuántos valores nulos hay en cada columna importante? Calcula
para: precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales,
metros_cuadrados_cubiertos, orientacion_cardinal, piso, cochera, antiguedad, estado, zona. Pista: Usa
COUNT(*) - COUNT(columna) para cada columna. Puedes usar CTEs para organizar.

Ejercicio 2.2: Porcentaje de nulos - ¿Qué porcentaje de registros tienen valores nulos en las columnas clave?
Calcula el porcentaje para: precio, expensas, ambientes, metros_cuadrados_totales,
metros_cuadrados_cubiertos, orientacion_cardinal, antiguedad. Pista: Usa ROUND((total - COUNT(columna)) *
100.0 / total, 2)

Ejercicio 2.3: Identificar columnas críticas - ¿Qué columnas tienen más del 50% de valores nulos? ¿Cuáles son
críticas para el análisis? Pista: Usa el resultado del ejercicio anterior y filtra con HAVING o WHERE

In [0]:
%sql
SELECT
    COUNT(*) - COUNT(precio) AS cant_null_precio,
    COUNT(*) - COUNT(expensas) AS cant_null_expensas,
    COUNT(*) - COUNT(tipo_de_operacion) AS cant_null_tipo_de_operacion,
    COUNT(*) - COUNT(moneda) AS cant_null_moneda,
    COUNT(*) - COUNT(ambientes) AS cant_null_ambientes,
    COUNT(*) - COUNT(metros_cuadrados_totales) AS cant_null_metros_cuadrados_totales,
    COUNT(*) - COUNT(metros_cuadrados_cubiertos) AS cant_null_metros_cuadrados_cubiertos,
    COUNT(*) - COUNT(orientacion_cardinal) AS cant_null_orientacion_cardinal,
    COUNT(*) - COUNT(piso) AS cant_null_piso,
    COUNT(*) - COUNT(cochera) AS cant_null_cochera,
    COUNT(*) - COUNT(antiguedad) AS cant_null_antiguedad,
    COUNT(*) - COUNT(estado) AS cant_null_estado,
    COUNT(*) - COUNT(zona) AS cant_null_zona
FROM bootcamp.bronze.properties_bronze

In [0]:

%sql
-- ============================================================
-- EDA 2.2: Porcentaje de nulos en columnas clave
-- ============================================================

WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp.bronze.properties_bronze
)
SELECT 
    ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as pct_nulos_precio,
    ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as pct_nulos_expensas,
    ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as pct_nulos_ambientes,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
FROM bootcamp.bronze.properties_bronze
CROSS JOIN totales t
GROUP BY t.total;

In [0]:
%sql
-- ============================================================
-- EDA 2.3: Identificar columnas con >50% de valores nulos
-- ============================================================
-- Columnas con >50% nulos son críticas: hay que decidir si eliminarlas o imputarlas

WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp.bronze.properties_bronze
),
pct_nulos AS (
    SELECT 
        ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as precio,
        ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as expensas,
        ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as ambientes,
        ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as metros_cuadrados_totales,
        ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as metros_cuadrados_cubiertos,
        ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as orientacion_cardinal,
        ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as antiguedad,
        ROUND((t.total - COUNT(piso)) * 100.0 / t.total, 2) as piso,
        ROUND((t.total - COUNT(cochera)) * 100.0 / t.total, 2) as cochera
    FROM bootcamp.bronze.properties_bronze
    CROSS JOIN totales t
    GROUP BY t.total
)
SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
    porcentaje_nulos FOR columna IN (
        precio, expensas, ambientes, 
        metros_cuadrados_totales, metros_cuadrados_cubiertos,
        orientacion_cardinal, antiguedad, piso, cochera
    )
)
WHERE porcentaje_nulos > 50
ORDER BY porcentaje_nulos DESC